In [93]:
# Kara sea individual case study:
import numpy as np
from numpy import ma

import matplotlib.pyplot as plt
from compiler.ast import flatten

import datetime, time
import pickle
import netCDF4

import matplotlib.pyplot as plt

import os, sys
sys.path.append('/home/cercache/users/atarasen/codes/python/')
sys.path.append('/home3/homedir7/perso/atarasen/codes/python/')
sys.path.append('/home3/homedir7/perso/atarasen/codes/python/collocation')

import nice_functions as nf
import nice_functions2 as nf2
reload(nf2)
#import ncdfView as nv

import plot_stereo
reload(plot_stereo)

import S1_open_process_and_visu as S1_open
reload (S1_open)

import collocation as colloc
reload(colloc)

import data_analysis as dt
reload(dt)

import buoy as buoy_module
reload (buoy_module)

import data_analysis
reload(data_analysis)

sys.path.append('/home3/homedir7/perso/atarasen/codes/python/posada/')
sys.path.append('/home3/homedir7/perso/atarasen/codes/python/cmod/')


from posada.Handlers.sar import SAR
from posada.Models.swath import Swath
from posada.Readers.safe import SAFEGeoTiffFile


from operator import add
import copy
from scipy import interpolate

import shapely
from shapely.geometry import Point, Polygon, MultiPolygon
from shapely.ops import cascaded_union, unary_union
from descartes.patch import PolygonPatch



# fpath = '/home/cercache/users/atarasen/data/Sentinel-1_analysis/Kara/'
# # fpathout = '/home/cercache/users/atarasen/data/Sentinel-1_analysis/Kara/mosaics/'
# fpathout = '/export/home/data/S1/Kara/mosaics/'

In [69]:
#Case 2:

# Regular grid result:

fname = "Kara_2014-10-03T003636_2014-10-03T102248_lat-3_lon-2_hv_1.pickle"
fpathout = '/export/home/data/S1/Kara/mosaics/'
with open(fpathout+fname, 'rb') as handle:
    total_dict = pickle.load(handle)
print  total_dict[('2014-10-03T003636', '2014-10-03T102248')].keys()
    

reg_s1 = total_dict[('2014-10-03T003636', '2014-10-03T102248')]['2014-10-03T102248']['s_1_regular']['hv']
reg_lat = total_dict[('2014-10-03T003636', '2014-10-03T102248')]['2014-10-03T102248']['lat_regular']
reg_lon = total_dict[('2014-10-03T003636', '2014-10-03T102248')]['2014-10-03T102248']['lon_regular']
c = total_dict[('2014-10-03T003636', '2014-10-03T102248')]['overlap_poly']


plt.figure(figsize=(24,24))
_, _, _, _, lat_center, lon_center, height_lats, width_lons = plot_stereo.zoom(reg_s1, reg_lat, reg_lon, \
             S1_name = '', pol='hv', case='Kara',\
             fig_title="result of interpolation",\
             buoy_case=None,\
             col_bar=True, col_map_name='bone')
plt.show()

['overlap_poly', '2014-10-03T003636', '2014-10-03T102248']


In [ ]:
####
# One-by-one-mosaics:

s1_path = '/home/cercache/users/atarasen/data/Sentinel-1_analysis/Kara/'
s1_names = ['S1A_EW_GRDH_1SDH_20141003T102248_20141003T102352_002664_002F76_222F.zip', 'S1A_EW_GRDH_1SDH_20141003T102352_20141003T102452_002664_002F76_43FC.zip']


regular_lat, regular_lon = colloc.regular_grid(c, deg_lat=3, deg_lon=2)

data_flat_total = []; mask_flat_total=[]; lat_flat_total=[]; lon_flat_total=[]
# plt.figure(figsize=(24,24))

for s1_name in s1_names:    
    sigma0, lat, lon180, _, elev_angle = S1_open.s1(s1_path+s1_name, 'hv', spacing_m=500, excl_bord=True)
    lon = plot_stereo.s1_lon360(lon180)
   
    
    if c.geom_type =='MultiPolygon':
        for one_poly in c:
            minx, miny, maxx, maxy = one_poly.bounds
            print minx, miny, maxx, maxy
            print np.min(np.min(lat)), np.min(np.min(lon)), np.max(np.max(lat)), np.max(np.max(lon))
            cond1 = np.logical_and(lat>minx, lat<maxx)
            cond2 = np.logical_and(lon>miny, lon<maxy)
            
            ind = np.where(np.logical_and(cond1, cond2))
            lat_flat = lat[zip(ind)].ravel();
            lon_flat = lon[zip(ind)].ravel();
            data_flat = sigma0[zip(ind)].ravel()

#             mask_flat = np.ones(len(lon_flat))
            
            
            points_collection =[Point(lat_i, lon_i) for lat_i, lon_i in zip(lat_flat, lon_flat)]

            for i_point, point in enumerate(points_collection):
                if one_poly.contains(point)==True:
                    data_flat_total.append(data_flat[i_point])
                    lat_flat_total.append(lat_flat[i_point])
                    lon_flat_total.append(lon_flat[i_point])
                    
#                     mask_flat[i_point] = 0
            print data_flat_total==[]        
#             mask_flat_total.append(mask_flat)
            
#             lat_flat_total.append(lat_flat)
#             lon_flat_total.append(lon_flat)

            plt.figure(figsize=(24,24))
            plt.scatter(lat_flat_total, lon_flat_total, data_flat_total)
            plt.show()

            
xy_to_interpolate=np.array([lat_flat_total, lon_flat_total]).T
s1_overlap_flat = np.array(data_flat_total)
regular_s1 = interpolate.griddata(xy_to_interpolate, s1_overlap_flat, (regular_lat, regular_lon), method='linear')

print regular_s1

# regular_s1 = ma.masked_where(np.isnan(regular_s1), regular_s1)


plt.figure(figsize=(24,24))          
plot_stereo.zoom_area(regular_s1, regular_lat, regular_lon, S1_name = '', case = 'Kara',\
                      width_lons=width_lons, height_lats=height_lats,\
                      lat_center=lat_center, lon_center=lon_center,\
                      fig_title='mosaic',\
                      pol = 'hv', col_bar=True, col_map_name='bone')
plt.show()

In [94]:
#Case 1:
#data from calculated regular grid
fname = "Kara_2014-10-16T025625_2014-10-16T110410_lat-3_lon-2_hv_1.pickle"
fpathout = '/export/home/data/S1/Kara/mosaics/'
with open(fpathout+fname, 'rb') as handle:
    total_dict = pickle.load(handle)
reg_s1 = total_dict[('2014-10-16T025625', '2014-10-16T110410')]['2014-10-16T110410']['s_1_regular']['hv']
reg_lat = total_dict[('2014-10-16T025625', '2014-10-16T110410')]['2014-10-16T110410']['lat_regular']
reg_lon = total_dict[('2014-10-16T025625', '2014-10-16T110410')]['2014-10-16T110410']['lon_regular']
c = total_dict[('2014-10-16T025625', '2014-10-16T110410')]['overlap_poly']

plt.figure(figsize=(24,24))
_, _, _, _, lat_center, lon_center, height_lats, width_lons = plot_stereo.zoom(reg_s1, reg_lat, reg_lon, \
             S1_name = '', pol='hv', case='Kara',\
             fig_title="result of interpolation",\
             buoy_case=None,\
             col_bar=True, col_map_name='bone')
plt.show()



# data from "Raw S1 images"
s1_path = '/home/cercache/users/atarasen/data/Sentinel-1_analysis/Kara/'
s1_names = ['S1A_EW_GRDH_1SDH_20141016T110514_20141016T110614_002854_003397_0432.zip', 'S1A_EW_GRDH_1SDH_20141016T110614_20141016T110709_002854_003397_FEDA.zip']


plt.figure(figsize=(24,24))

for s1_name in s1_names: 
    print s1_name
    sigma0, lat, lon180, _, elev_angle = S1_open.s1(s1_path+s1_name, 'hv', spacing_m=500, excl_bord=True)
    lon = plot_stereo.s1_lon360(lon180)
    
    lat_flat = lat.ravel(); lon_flat = lon.ravel()

    if c.geom_type =='MultiPolygon':
        for one_poly in c:
            minx, miny, maxx, maxy = c.bounds
            print minx, miny, maxx, maxy
            print np.min(np.min(lon)), np.min(np.min(lat)), np.max(np.max(lon)),  np.max(np.max(lat))
            cond1 = np.logical_and(lat<minx, lat>maxx)
            cond2 = np.logical_and(lon<miny, lon>maxy)
            
            ind = np.where(np.logical_and(cond1, cond2))
            lat_flat = lat[zip(ind)].ravel();
            lon_flat = lon[zip(ind)].ravel();
            print lat_flat
            mask_flat = np.ones(len(lon_flat))
            
            points_collection =[Point(lat_i, lon_i) for lat_i, lon_i in zip(lat_flat, lon_flat)]

            for i_point, point in enumerate(points_collection):
                if one_poly.contains(point)==True:
                    print "data inside the "
                    mask_flat[i_point] = 0
#         else:
#             if c.contains(point)==True:
#                     mask_flat[i_point] = 0
    
    
#     plot_stereo.zoom_area(sigma0, lat, lon, S1_name = s1_name, case = 'Kara',\
#                           width_lons=width_lons, height_lats=height_lats,\
#                           lat_center=lat_center, lon_center=lon_center,\
#                           fig_title='mosaic',\
#                           pol = 'hv', col_bar=True, col_map_name='bone')
# plt.show()

KeyError: 'overlap_poly'